Language: **English (default)** | [Polski](lab.ipynb)

# Lab 06. Designing, training, and analyzing a CNN on KMNIST

This companion to [`README.md`](README.md) uses KMNIST instead of
Fashion-MNIST. The experiment examines how design choices within a CNN
affect quality, parameter count, and computational cost.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Data, class distribution, and tensor shapes

In [ ]:
import contextlib
import io

_REPO_ROOT = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "pyproject.toml").is_file()
    ),
    Path.cwd(),
)
DATA_CACHE_DIR = (
    Path(os.environ.get("DEEP_LEARNING_DATA_CACHE", _REPO_ROOT / "data_cache"))
    .expanduser()
    .resolve()
)

# Downloading prints progress (a tqdm bar, by default on stderr) every few
# percent -- in a notebook that is just noise, so we silence both streams
# here (the file and its MD5 are verified by torchvision regardless of
# what it prints to the screen).
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    train_full = torchvision.datasets.KMNIST(
        root=DATA_CACHE_DIR, train=True, download=True, transform=T.ToTensor()
    )
    test_full = torchvision.datasets.KMNIST(
        root=DATA_CACHE_DIR, train=False, download=True, transform=T.ToTensor()
    )

class_names = train_full.classes
print(f"train_full: {len(train_full)} examples, test_full: {len(test_full)} examples")
print(f"shape of one image: {tuple(train_full[0][0].shape)}")
print("classes (kuzushiji characters):", class_names)

fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
for ax, i in zip(axes, range(8)):
    img, label = train_full[i]
    ax.imshow(img[0], cmap="gray")
    ax.set_title(class_names[label], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

**Task.** Use `torch.bincount` on `train_full.targets` to count the examples
from each of the 10 classes and determine whether the training set is
approximately balanced.

In [ ]:
# TODO: count the number of examples of each of the 10 classes in
# train_full.targets (torch.bincount) and print it
raise NotImplementedError("TODO: class distribution")

_(Answer space.)_

## 2. Train/val/test split and leak-free normalization

KMNIST already has a separate test set (`test_full`, a different file from
the training one). A validation set must be taken **from the training data**
with no overlapping indices.

In [ ]:
def make_split(n_total, n_train, n_val, seed):
    """Returns (train_idx, val_idx) -- disjoint indices from [0, n_total)."""
    # TODO:
    # 1. use torch.randperm(n_total, generator=...) with the given seed
    # 2. train_idx = the first n_train indices of the permutation
    # 3. val_idx = the next n_val indices (NOT overlapping with train_idx)
    raise NotImplementedError("TODO: make_split")

In [ ]:
N_TRAIN, N_VAL, N_TEST = 5000, 1000, 1500
SPLIT_SEED = 0

train_idx, val_idx = make_split(len(train_full), N_TRAIN, N_VAL, SPLIT_SEED)
assert len(set(train_idx.tolist()) & set(val_idx.tolist())) == 0, "train/val overlap!"

g = torch.Generator().manual_seed(SPLIT_SEED)
test_idx = torch.randperm(len(test_full), generator=g)[:N_TEST]

X_train_raw = torch.stack([train_full[i][0] for i in train_idx])
y_train = torch.stack([torch.tensor(train_full[i][1]) for i in train_idx])
X_val_raw = torch.stack([train_full[i][0] for i in val_idx])
y_val = torch.stack([torch.tensor(train_full[i][1]) for i in val_idx])
X_test_raw = torch.stack([test_full[i][0] for i in test_idx])
y_test = torch.stack([torch.tensor(test_full[i][1]) for i in test_idx])

print(
    f"X_train={tuple(X_train_raw.shape)}  X_val={tuple(X_val_raw.shape)}  "
    f"X_test={tuple(X_test_raw.shape)}"
)

The `normalize_with_train_stats` function should calculate the pixel mean
and standard deviation **only** on `X_train_raw`. Apply the same statistics
to all three sets without recomputing them.

In [ ]:
def normalize_with_train_stats(X_train_raw, X_val_raw, X_test_raw):
    """Returns (mean, std, X_train, X_val, X_test). Statistics ONLY from X_train_raw."""
    # TODO:
    # 1. compute mean, std EXCLUSIVELY from X_train_raw (scalars, whole tensors)
    # 2. normalize X_train_raw, X_val_raw, X_test_raw with THESE SAME
    #    statistics (do not recompute separately for val/test!)
    # 3. return (mean, std, X_train, X_val, X_test)
    raise NotImplementedError("TODO: leak-free normalization")

In [ ]:
pixel_mean, pixel_std, X_train, X_val, X_test = normalize_with_train_stats(
    X_train_raw, X_val_raw, X_test_raw
)
X_train, y_train = X_train.to(device), y_train.to(device)
X_val, y_val = X_val.to(device), y_val.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

print(f"pixel mean (train): {pixel_mean.item():.4f}  std: {pixel_std.item():.4f}")
print(
    f"after normalization, X_train: mean={X_train.mean().item():.4f} std={X_train.std().item():.4f}"
)
assert abs(X_train.mean().item()) < 0.05
assert abs(X_train.std().item() - 1.0) < 0.05
print("OK: X_train after normalization has mean ~0 and std ~1.")

## 3. Baseline model (provided)

The simple linear classifier is not a CNN and does not use the image's
two-dimensional structure. Its result provides a baseline for comparison.

In [ ]:
def train_eval(model_cls, epochs, lr=0.001, batch_size=64, seed=0):
    """Constructs model_cls() AFTER setting the seed (controls weight
    init, not just batch order), trains on (X_train, y_train), returns
    (model, metrics)."""
    import time

    torch.manual_seed(seed)
    model = model_cls().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    n = X_train.shape[0]

    start = time.time()
    for _epoch in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i : i + batch_size]
            opt.zero_grad()
            out = model(X_train[idx])
            loss = loss_fn(out, y_train[idx])
            loss.backward()
            opt.step()
    train_time = time.time() - start

    model.eval()
    with torch.no_grad():
        val_acc = (model(X_val).argmax(1) == y_val).float().mean().item()
        start = time.time()
        test_logits = model(X_test)
        infer_time = time.time() - start
    test_acc = (test_logits.argmax(1) == y_test).float().mean().item()
    train_acc = (model(X_train).argmax(1) == y_train).float().mean().item()
    n_params = sum(p.numel() for p in model.parameters())

    return model, {
        "train_acc": train_acc,
        "val_acc": val_acc,
        "test_acc": test_acc,
        "train_time_s": train_time,
        "infer_time_s": infer_time,
        "n_params": n_params,
    }


class Baseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 10))

    def forward(self, x):
        return self.net(x)


baseline_model, baseline_res = train_eval(Baseline, epochs=5, seed=0)
print("Baseline:", baseline_res)

## 4. CNN design, Variant 1

Design `CNNVariant1` with `nn.Conv2d`, an activation function, and optionally
`nn.MaxPool2d`. The model must accept `(N, 1, 28, 28)` and return `(N, 10)`.

In [ ]:
class CNNVariant1(nn.Module):
    """Your design: choose the number of convolutional layers, the number
    of filters, the kernel size, whether to use pooling. The model must
    accept (N, 1, 28, 28) and return (N, 10)."""

    def __init__(self):
        super().__init__()
        # TODO: define layers (nn.Conv2d, nn.MaxPool2d, nn.Linear, ...)
        raise NotImplementedError("TODO: CNNVariant1 architecture")

    def forward(self, x):
        # TODO: chain layers with activations (e.g. torch.relu), flatten
        # (x.flatten(1)) before the final Linear layer
        raise NotImplementedError("TODO: CNNVariant1 forward")

**Required shape verification.** Pass a sample batch through the model and
check the tensor shape after every layer. This catches an incorrect input
size for `nn.Linear` after `flatten()`.

In [ ]:
def print_shapes(model, input_shape=(2, 1, 28, 28)):
    x = torch.zeros(*input_shape)
    hooks = []

    def hook(module, _inp, out):
        print(f"{module.__class__.__name__:>12s} -> {tuple(out.shape)}")

    for layer in model.children():
        hooks.append(layer.register_forward_hook(hook))
    print(f"{'input':>12s} -> {tuple(x.shape)}")
    with torch.no_grad():
        model(x)
    for h in hooks:
        h.remove()


print_shapes(CNNVariant1())

**Task.** Describe and justify the selected number of layers and filters,
kernel size, and use of pooling.

_(Answer space.)_

## 5. Training and validating Variant 1

In [ ]:
variant1_model, variant1_res = train_eval(CNNVariant1, epochs=5, seed=0)
print("Variant 1:", variant1_res)

## 6. CNN design, Variant 2

The second variant must differ from the first in at least one specific and
justified aspect.

**Task before writing code.** State the specific hypothesis tested by the
introduced change.

_(Answer space.)_

In [ ]:
class CNNVariant2(nn.Module):
    """Your second design -- must differ from Variant 1 in a specific,
    named way (e.g. a different number of filters, kernel, presence of
    pooling)."""

    def __init__(self):
        super().__init__()
        # TODO: define layers, different from CNNVariant1
        raise NotImplementedError("TODO: CNNVariant2 architecture")

    def forward(self, x):
        # TODO: forward for Variant 2
        raise NotImplementedError("TODO: CNNVariant2 forward")

In [ ]:
print_shapes(CNNVariant2())

## 7. Comparing the baseline and two CNN variants

Train all three models with at least two random seeds. The summary should
include validation and test accuracy, parameter count, and training and
inference time. The test set may be used for the first time only here.

In [ ]:
# TODO:
# 1. for each of the three models (Baseline, CNNVariant1, CNNVariant2) and
#    each of at least 2 seeds (e.g. [0, 1]): call
#    train_eval(CLASS, epochs=5, seed=seed) -- PASS THE CLASS, not an
#    instance (model()); train_eval constructs the model AFTER setting
#    the seed, otherwise the seed does not control weight initialization
# 2. store the results (val_acc, test_acc, n_params, train_time_s, ...) so
#    you can summarize them per model (e.g. a dict model_name -> list of results)
# 3. store the TRAINED model for seed=0 of each architecture -- needed in
#    section 8 for the confusion matrix
# 4. print a summary: for each model, the averaged val_acc/test_acc across
#    seeds, together with parameter count and training time
raise NotImplementedError("TODO: compare the three models")

## 8. Confusion matrix and error analysis

Select the model with the best **validation** accuracy from section 7.

In [ ]:
# TODO:
# 1. choose the model with the best AVERAGE validation accuracy (across
#    seeds) from all_results/trained_models in section 7
# 2. compute this model's predictions on X_test (model.eval(), torch.no_grad())
# 3. build a 10x10 confusion matrix (rows=true class, columns=predicted) --
#    either by hand in a loop over (y_true, y_pred) or torch.zeros(10,10) +
#    incrementing
# 4. draw the confusion matrix (plt.imshow) with class labels on both axes
# 5. find the indices of misclassified test examples, display 6-8 of them
#    (image + true/predicted label in the title)
raise NotImplementedError("TODO: confusion matrix and error analysis")

_(Answer space.)_

## 9. Conclusions

Answers should be supported by actual results from sections 7 and 8.

**1. Which CNN variant won on validation? Is it the same one that won on the test set?**

_(Answer space.)_

**2. Did the variant with more parameters train faster or slower? Consistent with the intuition "fewer parameters = fewer computations," or does it contradict it?**

_(Answer space.)_

**3. Does higher training accuracy for the best model demonstrate good generalization? Which results support the assessment?**

_(Answer space.)_

**4. Which limitation of this experiment prevents treating Variant X as universally better?**

_(Answer space.)_